In [244]:
import pandas as pd
import numpy as np
from datetime import datetime

In [279]:
repert = 'c:\\Users\\Amandine\\Downloads\\'

nom_feuille = "KBRW_ORDER_LINE"
colonnes = ['SOURCE_LOCATION','TIMESTAMP','REVISED_EFD_MIN','REVISED_EFD_MAX','DELIVERY_STATUS','DELIVERY_STATUS_DATE']
datalake = pd.read_excel(repert + 'Extract ALL_ Datalake.xlsx',sheet_name=nom_feuille,usecols=colonnes)

nom_feuille2 = "AN_APP031_TO_BE_SHIPPED_ARCHIVE"
colonnes2 = ['RDC','WEEK','TO_BE_RECEIVED_QTY','ARCHIVE_TMSTP']
visibilite2 = pd.read_excel(repert + 'Extract ALL_ Archive.xlsx',sheet_name=nom_feuille2,usecols=colonnes2)

nom_feuille3 = "ARCHIVE_FORECAST_EXPORTED_WEEK"
colonnes3 = ['FORECAST_LEVEL','WEEK','QTY','EXTRACT_TMSTP']
forecast = pd.read_excel(repert + 'Extract ALL_ Archive.xlsx',sheet_name=nom_feuille3,usecols=colonnes3)

nom_feuille4="PDP_PDL_QTY_ARCHIVE"
colonnes4 = ['ATELIER','WEEK','VALUE','EXTRACT_TMSTP']
PDL = pd.read_excel(repert + 'Extract ALL_ Archive.xlsx',sheet_name=nom_feuille4,usecols=colonnes4)

In [377]:
nom_feuille5="AN_APP010_EXP_NEED_ARCHIVE"
colonnes5=['CDC','WEEK','VALUE','TIMESTAMP']
need = pd.read_excel(repert + 'Extract ALL_ Archive.xlsx',sheet_name=nom_feuille5,usecols=colonnes5)

In [386]:
datalake['DELIVERY_STATUS_DATE'] = pd.to_datetime(datalake['DELIVERY_STATUS_DATE'], errors='coerce', format='%Y-%m-%d')
datalake['REVISED_EFD_MAX'] = pd.to_datetime(datalake['REVISED_EFD_MAX'], errors='coerce', format='%Y-%m-%d')
datalake['REVISED_EFD_MIN'] = pd.to_datetime(datalake['REVISED_EFD_MIN'], errors='coerce', format='%Y-%m-%d')
datalake['TIMESTAMP'] = pd.to_datetime(datalake['TIMESTAMP'], errors='coerce', format='%Y-%m-%d')

datalake['DELIVERY_STATUS_DATE'] = datalake['DELIVERY_STATUS_DATE'].dt.date.astype('datetime64[ns]')
datalake['REVISED_EFD_MAX'] = datalake['REVISED_EFD_MAX'].dt.date.astype('datetime64[ns]')
datalake['REVISED_EFD_MIN'] = datalake['REVISED_EFD_MIN'].dt.date.astype('datetime64[ns]')
datalake['TIMESTAMP'] = datalake['TIMESTAMP'].dt.date.astype('datetime64[ns]')

visibilite2['ARCHIVE_TMSTP'] = pd.to_datetime(visibilite2['ARCHIVE_TMSTP'], errors='coerce')
visibilite2['ARCHIVE_TMSTP'] = visibilite2['ARCHIVE_TMSTP'].dt.date.astype('datetime64[ns]')

forecast['EXTRACT_TMSTP'] = pd.to_datetime(forecast['EXTRACT_TMSTP'], errors='coerce')
forecast['EXTRACT_TMSTP'] = forecast['EXTRACT_TMSTP'].dt.date.astype('datetime64[ns]')

PDL['EXTRACT_TMSTP'] = pd.to_datetime(PDL['EXTRACT_TMSTP'], errors='coerce')
PDL['EXTRACT_TMSTP'] = PDL['EXTRACT_TMSTP'].dt.date.astype('datetime64[ns]')

need['TIMESTAMP'] = pd.to_datetime(need['TIMESTAMP'], errors='coerce')
need['TIMESTAMP'] = need['TIMESTAMP'].dt.date.astype('datetime64[ns]')

In [387]:
datalake['DifferenceEnJours'] = (datalake['DELIVERY_STATUS_DATE'] - datalake['REVISED_EFD_MAX']).dt.days #à utiliser seulement pour les retards
datalake['DifferenceEnJours2'] = (datalake['REVISED_EFD_MIN']-datalake['DELIVERY_STATUS_DATE']).dt.days #à utiliser seulement pour les avances

In [503]:
print(datalake[(datalake['DifferenceEnJours2'] > 6) & (datalake['DELIVERY_STATUS']=='delivered') & (datalake['TIMESTAMP'] > pd.to_datetime('2023-10-15'))])

       SOURCE_LOCATION  TIMESTAMP REVISED_EFD_MIN REVISED_EFD_MAX  \
102584             E2Z 2023-12-17      2024-01-18      2024-02-03   

       DELIVERY_STATUS DELIVERY_STATUS_DATE  DifferenceEnJours  \
102584       delivered           2023-12-20              -45.0   

        DifferenceEnJours2  
102584                29.0  


In [573]:
#Variables
nb_jours_acceptés=6
seuil_visibilité = 20
seuil_forecast = 10
seuil_PDL = 10
seuil_need = 10

In [574]:
def numero_semaine(date):
    date_obj = np.datetime64(date, 'D').astype(datetime)
    numero_semaine = date_obj.isocalendar()[1]
    return 'W2023' + str(numero_semaine)

def is_late(row):
    if row['DifferenceEnJours'] <= nb_jours_acceptés:
        return False
    else:
        return True
    
def is_in_advance(row):
    if row['DifferenceEnJours2'] <= nb_jours_acceptés:
        return False
    else:
        return True
    
def nombre_commandes_livrees(datalake):
    nb_commandes_livrées = (datalake['DELIVERY_STATUS']=='delivered').sum()
    return nb_commandes_livrées
    #print(f"Le nombre de commandes livrées est : {nb_commandes_livrées}")

def donnees_timestamp(timestamp): #vérifier si on a la date de prise de commande dans les données d'archive
    est_presente = (visibilite2['ARCHIVE_TMSTP'] == timestamp).any()
    if est_presente:
        return True
    else:
        return False

def donnees_week(revised_efd_max,dataframe): #vérifier si on a la semaine de livraison prévue dans les données d'archive
    est_presente = (dataframe['WEEK'] == numero_semaine(revised_efd_max)).any()
    if est_presente:
        return True
    else:
        return False

def donnees_timestamp_extract(timestamp,dataframe):
    est_presente = (dataframe['EXTRACT_TMSTP'] == timestamp).any()
    if est_presente:
        return True
    else:
        return False

def donnees_timestamp_timestamp(timestamp):
    est_presente = (need['TIMESTAMP'] == timestamp).any()
    if est_presente:
        return True
    else:
        return False

def localisation(RDC):
    if RDC=='NU8' or RDC=='NU9':
        return 'CHINA'
    elif RDC=='EUZ':
        return 'MIDDLE EAST'
    elif RDC=='U2R' or RDC=='U3R':
        return 'NORTH AMERICA'
    elif RDC=='EU2':
        return 'EUROPE W/O MIDDLE EAST'
    elif RDC=='N71':
        return '1'
    elif RDC=='NF8':
        return '2'
    elif RDC=='EUR':
        return '3'

def diff_jours_min(timestamp,dataframe):
    date_plus_proche = dataframe['EXTRACT_TMSTP'].iloc[(dataframe['EXTRACT_TMSTP'] - timestamp).abs().idxmin()]
    min = pd.to_datetime(date_plus_proche)
    diff=(min-timestamp).days
    if abs(diff)<=6:
        return min

def diff_jours_min2(timestamp):
    date_plus_proche = need['TIMESTAMP'].iloc[(need['TIMESTAMP'] - timestamp).abs().idxmin()]
    min = pd.to_datetime(date_plus_proche)
    diff=(min-timestamp).days
    if abs(diff)<=6:
        return min



In [575]:
def variation_visibility(timestamp, revised_efd_max, RDC):
    timestamp_check = donnees_timestamp(timestamp)
    week_check = donnees_week(revised_efd_max,visibilite2)
    compteur = 0
    if timestamp_check and week_check:
        date = timestamp + np.timedelta64(7, 'D')
        if donnees_timestamp(date):
            while date < revised_efd_max - np.timedelta64(6, 'D'):
                compteur = 1
                condition = (visibilite2['ARCHIVE_TMSTP'] == timestamp) & (visibilite2['RDC'] == RDC) & (visibilite2['WEEK'] == numero_semaine(revised_efd_max))
                condition2 = (visibilite2['ARCHIVE_TMSTP'] == date) & (visibilite2['RDC'] == RDC) & (visibilite2['WEEK'] == numero_semaine(revised_efd_max))

                if 100*abs(visibilite2.loc[condition, 'TO_BE_RECEIVED_QTY'].values[0]-visibilite2.loc[condition2, 'TO_BE_RECEIVED_QTY'].values[0])/visibilite2.loc[condition, 'TO_BE_RECEIVED_QTY'].values[0]> seuil_visibilité:
                    return True
            
                date += np.timedelta64(7, 'D')
                if not donnees_timestamp(date):
                    date += np.timedelta64(7, 'D')
            if compteur ==1 :
                return False
   


In [576]:
def variation_forecast(timestamp, revised_efd_max, RDC):
    extract=diff_jours_min(timestamp,forecast)
    timestamp_check = donnees_timestamp_extract(extract,forecast)
    week_check = donnees_week(revised_efd_max,forecast)
    CDC=localisation(RDC)
    compteur=0
    if timestamp_check and week_check:
        date = extract + np.timedelta64(7, 'D')
        date_extract=diff_jours_min(date,forecast)
        if donnees_timestamp_extract(date_extract,forecast):
            while date_extract < revised_efd_max - np.timedelta64(6, 'D'):
                if CDC=='1':
                    L=['JAPAN','KOREA (DOM)','NORTH ASIA','SOUTH ASIA W/O KDOM']
                    for i in range(len(L)):
                        condition = (forecast['EXTRACT_TMSTP'] == extract) & (forecast['FORECAST_LEVEL'] == L[i]) & (forecast['WEEK'] == numero_semaine(revised_efd_max))
                        condition2 = (forecast['EXTRACT_TMSTP'] == date_extract) & (forecast['FORECAST_LEVEL'] == L[i]) & (forecast['WEEK'] == numero_semaine(revised_efd_max))
                        if 100*abs(forecast.loc[condition, 'QTY'].values[0]-forecast.loc[condition2, 'QTY'].values[0])/forecast.loc[condition, 'QTY'].values[0]> seuil_forecast:
                            return True
                    
                elif CDC=='2':
                    L=['CHINA','KOREA (DOM)','NORTH ASIA']
                    for i in range(len(L)):
                        condition = (forecast['EXTRACT_TMSTP'] == extract) & (forecast['FORECAST_LEVEL'] == L[i]) & (forecast['WEEK'] == numero_semaine(revised_efd_max))
                        condition2 = (forecast['EXTRACT_TMSTP'] == date_extract) & (forecast['FORECAST_LEVEL'] == L[i]) & (forecast['WEEK'] == numero_semaine(revised_efd_max))
                        if 100*abs(forecast.loc[condition, 'QTY'].values[0]-forecast.loc[condition2, 'QTY'].values[0])/forecast.loc[condition, 'QTY'].values[0]> seuil_forecast:
                            return True
                            
                elif CDC=='3':
                    L=['LATIN AMERICA','EUROPE W/O MIDDLE EAST']
                    for i in range(len(L)):
                        condition = (forecast['EXTRACT_TMSTP'] == extract) & (forecast['FORECAST_LEVEL'] == L[i]) & (forecast['WEEK'] == numero_semaine(revised_efd_max))
                        condition2 = (forecast['EXTRACT_TMSTP'] == date_extract) & (forecast['FORECAST_LEVEL'] == L[i]) & (forecast['WEEK'] == numero_semaine(revised_efd_max))
                        if 100*abs(forecast.loc[condition, 'QTY'].values[0]-forecast.loc[condition2, 'QTY'].values[0])/forecast.loc[condition, 'QTY'].values[0]> seuil_forecast:
                            return True

                else:
                    condition = (forecast['EXTRACT_TMSTP'] == extract) & (forecast['FORECAST_LEVEL'] == CDC) & (forecast['WEEK'] == numero_semaine(revised_efd_max))
                    condition2 = (forecast['EXTRACT_TMSTP'] == date_extract) & (forecast['FORECAST_LEVEL'] == CDC) & (forecast['WEEK'] == numero_semaine(revised_efd_max))
                    if 100*abs(forecast.loc[condition, 'QTY'].values[0]-forecast.loc[condition2, 'QTY'].values[0])/forecast.loc[condition, 'QTY'].values[0]> seuil_forecast:
                        return True
                
                date += np.timedelta64(7, 'D')
                date_extract=diff_jours_min(date,forecast)
                if not donnees_timestamp_extract(date_extract,forecast):
                    date += np.timedelta64(7, 'D')
                    date_extract=diff_jours_min(date,forecast)
 
                

In [577]:
def variation_pdl(timestamp, revised_efd_max, RDC):
    extract=diff_jours_min(timestamp,PDL)
    timestamp_check = donnees_timestamp_extract(extract,PDL)
    week_check = donnees_week(revised_efd_max,PDL)
    if timestamp_check and week_check:
        date = extract + np.timedelta64(7, 'D')
        date_extract=diff_jours_min(date,PDL)
        if donnees_timestamp_extract(date_extract,PDL):
            while date_extract < revised_efd_max - np.timedelta64(6, 'D'):
                if RDC=='U2R' or RDC=='U3R' or RDC=='UTR':
                    condition = []
                    condition2 = []
                    somme1=0
                    somme2=0
                    c = (PDL['EXTRACT_TMSTP'] == extract) & (PDL['ATELIER'] == 'Ardeche') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == extract) & (PDL['ATELIER'] == 'Atelier72') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == extract) & (PDL['ATELIER'] == 'Conde') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == extract) & (PDL['ATELIER'] == 'Firenze') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == extract) & (PDL['ATELIER'] == 'MMD') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == extract) & (PDL['ATELIER'] == 'San Dimas 2') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == extract) & (PDL['ATELIER'] == 'Texas Rancho') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == date_extract) & (PDL['ATELIER'] == 'Ardeche') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition2.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == date_extract) & (PDL['ATELIER'] == 'Atelier72') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition2.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == date_extract) & (PDL['ATELIER'] == 'Conde') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition2.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == date_extract) & (PDL['ATELIER'] == 'Firenze') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition2.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == date_extract) & (PDL['ATELIER'] == 'MMD') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition2.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == date_extract) & (PDL['ATELIER'] == 'San Dimas 2') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition2.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == date_extract) & (PDL['ATELIER'] == 'Texas Rancho') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition2.append(c)
                    for i in range(len(condition)):
                        somme1+=PDL.loc[condition[i], 'VALUE'].values[0]
                    for j in range(len(condition2)):
                        somme2+=PDL.loc[condition2[i], 'VALUE'].values[0]
                    if 100*abs(somme1-somme2)/somme1> seuil_PDL:
                        return True
                
                else:
                    condition = []
                    condition2 = []
                    somme1=0
                    somme2=0
                    c = (PDL['EXTRACT_TMSTP'] == extract) & (PDL['ATELIER'] == 'Ardeche') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == extract) & (PDL['ATELIER'] == 'Atelier72') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == extract) & (PDL['ATELIER'] == 'Conde') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == extract) & (PDL['ATELIER'] == 'Firenze') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == extract) & (PDL['ATELIER'] == 'MMD') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == date_extract) & (PDL['ATELIER'] == 'Ardeche') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition2.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == date_extract) & (PDL['ATELIER'] == 'Atelier72') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition2.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == date_extract) & (PDL['ATELIER'] == 'Conde') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition2.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == date_extract) & (PDL['ATELIER'] == 'Firenze') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition2.append(c)
                    c = (PDL['EXTRACT_TMSTP'] == date_extract) & (PDL['ATELIER'] == 'MMD') & (PDL['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition2.append(c)
                    for i in range(len(condition)):
                        somme1+=PDL.loc[condition[i], 'VALUE'].values[0]
                    for j in range(len(condition2)):
                        somme2+=PDL.loc[condition2[i], 'VALUE'].values[0]
                    if 100*abs(somme1-somme2)/somme1> seuil_PDL:
                        return True

                date += np.timedelta64(7, 'D')
                date_extract=diff_jours_min(date,PDL)
                if not donnees_timestamp_extract(date_extract,PDL):
                    date += np.timedelta64(7, 'D')
                    date_extract=diff_jours_min(date,PDL)

In [578]:
def variation_need(timestamp, revised_efd_max, RDC):
    extract=diff_jours_min2(timestamp)
    timestamp_check = donnees_timestamp_timestamp(extract)
    week_check = donnees_week(revised_efd_max,need)
    if timestamp_check and week_check:
        date = extract + np.timedelta64(7, 'D')
        date_extract=diff_jours_min2(date)
        if donnees_timestamp_timestamp(date_extract):
            while date_extract < revised_efd_max - np.timedelta64(6, 'D'):
                if RDC=='U2R' or RDC=='U3R' or RDC=='UTR':
                    condition = []
                    condition2 = []
                    somme1=0
                    somme2=0
                    c = (need['TIMESTAMP'] == extract) & (need['CDC'] == 'CDC_CEN') & (need['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition.append(c)
                    c = (need['TIMESTAMP'] == extract) & (need['CDC'] == 'CDC_USA') & (need['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition.append(c)
                    c = (need['TIMESTAMP'] == date_extract) & (need['CDC'] == 'CDC_CEN') & (need['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition2.append(c)
                    c = (need['TIMESTAMP'] == date_extract) & (need['CDC'] == 'CDC_USA') & (need['WEEK'] == numero_semaine(revised_efd_max))
                    if c.any() :
                        condition2.append(c)
                    for i in range(len(condition)):
                        somme1+=need.loc[condition[i], 'VALUE'].values[0]
                    for j in range(len(condition2)):
                        somme2+=need.loc[condition2[i], 'VALUE'].values[0]
                    if 100*abs(somme1-somme2)/somme1> seuil_need:
                        return True
                else:
                    condition = (need['TIMESTAMP'] == extract) & (need['CDC'] == 'CDC_CEN') & (need['WEEK'] == numero_semaine(revised_efd_max))
                    condition2 = (need['TIMESTAMP'] == date_extract) & (need['CDC'] == 'CDC_CEN') & (need['WEEK'] == numero_semaine(revised_efd_max))
                    if 100*abs(need.loc[condition, 'VALUE'].values[0]-need.loc[condition2, 'VALUE'].values[0])/need.loc[condition, 'VALUE'].values[0]>seuil_need:
                        return True
                date += np.timedelta64(7, 'D')
                date_extract=diff_jours_min2(date)
                if not donnees_timestamp_timestamp(date_extract):
                    date += np.timedelta64(7, 'D')
                    date_extract=diff_jours_min2(date)
    

In [579]:
#Pourcentages 
global nb_commandes_en_avance
global nb_commandes_en_retard
global nb_commandes_mauvais_lt     
global visibility_change 
global visibility_non_change 
global forecast_change
global pdl_change
global need_change
nb_commandes_en_avance=0
nb_commandes_en_retard=0
nb_commandes_mauvais_lt=nb_commandes_en_avance+nb_commandes_en_retard
visibility_change = 0
visibility_non_change = 0
forecast_change = 0
pdl_change = 0
need_change = 0 

In [580]:
# Fonction personnalisée pour itérer sur les lignes
def iterer_sur_lignes(row):
    global nb_commandes_en_avance
    global nb_commandes_en_retard
    global nb_commandes_mauvais_lt   
    global visibility_change 
    global visibility_non_change 
    global forecast_change
    global pdl_change
    global need_change
    if row['DELIVERY_STATUS']=='delivered':
        if is_late(row):
            nb_commandes_en_retard+=1
            if variation_visibility(row['TIMESTAMP'], row['REVISED_EFD_MAX'], row['SOURCE_LOCATION']):
                visibility_change+=1
                if variation_forecast(row['TIMESTAMP'], row['REVISED_EFD_MAX'], row['SOURCE_LOCATION']):
                    forecast_change+=1
                else:
                    if variation_pdl(row['TIMESTAMP'], row['REVISED_EFD_MAX'], row['SOURCE_LOCATION']):
                        pdl_change+=1
                        if variation_need(row['TIMESTAMP'], row['REVISED_EFD_MAX'], row['SOURCE_LOCATION']):
                            need_change+=1
            elif variation_visibility(row['TIMESTAMP'], row['REVISED_EFD_MAX'], row['SOURCE_LOCATION'])==False:
                visibility_non_change+=1
        elif is_in_advance(row):
            nb_commandes_en_avance+=1
            if variation_visibility(row['TIMESTAMP'], row['REVISED_EFD_MIN'], row['SOURCE_LOCATION']):
                visibility_change+=1
            elif variation_visibility(row['TIMESTAMP'], row['REVISED_EFD_MIN'], row['SOURCE_LOCATION'])==False:
                visibility_non_change+=1
    nb_commandes_mauvais_lt=nb_commandes_en_avance+nb_commandes_en_retard

In [581]:
datalake.apply(iterer_sur_lignes, axis=1)

0         None
1         None
2         None
3         None
4         None
          ... 
106362    None
106363    None
106364    None
106365    None
106366    None
Length: 106367, dtype: object

In [582]:
nb_commandes_en_retard

1745

In [583]:
nb_commandes_en_avance

2138

In [584]:
nb_commandes_mauvais_lt     

3883

In [585]:
visibility_change

73

In [586]:
visibility_non_change

7

In [587]:
forecast_change

66

In [588]:
pdl_change

7

In [589]:
need_change

6

In [590]:
nb_commandes_livrees = nombre_commandes_livrees(datalake)
pourcentage_mauvais_lt = 100*nb_commandes_mauvais_lt/nb_commandes_livrees
pourcentage_avance = 100*nb_commandes_en_avance/nb_commandes_mauvais_lt
pourcentage_retard = 100*nb_commandes_en_retard/nb_commandes_mauvais_lt
nb_donnees = visibility_change+visibility_non_change
pourcentage_donnees = 100*nb_donnees/nb_commandes_mauvais_lt
pourcentage_visibility = 100*visibility_change/nb_donnees
pourcentage_forecast = 100*forecast_change/visibility_change
pourcentage_pdl = 100*pdl_change/visibility_change
pourcentage_need = 100*need_change/pdl_change
print(f"Le nombre de commandes livrées est : {nb_commandes_livrees}")
print(f"Le pourcentage de commandes livrées en avance ou en retard est : {pourcentage_mauvais_lt}")
print(f"Le pourcentage de commandes en avance est : {pourcentage_avance}")
print(f"Le pourcentage de commandes en retard est : {pourcentage_retard}")
print(f"Le pourcentage de données d'archive qu'on possède sur les commandes avec un mauvais leadtime est : {pourcentage_donnees}")
print(f"Le pourcentage de changement de visibilité est : {pourcentage_visibility}")
print(f"Le pourcentage de changement de forecast est : {pourcentage_forecast}")
print(f"Le pourcentage de changement de PDL est : {pourcentage_pdl}")
print(f"Le pourcentage de changement de need est : {pourcentage_need}")

Le nombre de commandes livrées est : 20437
Le pourcentage de commandes livrées en avance ou en retard est : 18.999853207417917
Le pourcentage de commandes en avance est : 55.06052021632758
Le pourcentage de commandes en retard est : 44.93947978367242
Le pourcentage de données d'archive qu'on possède sur les commandes avec un mauvais leadtime est : 2.0602626834921454
Le pourcentage de changement de visibilité est : 91.25
Le pourcentage de changement de forecast est : 90.41095890410959
Le pourcentage de changement de PDL est : 9.58904109589041
Le pourcentage de changement de need est : 85.71428571428571


0.0
